# Environment and Qwen 27B compatibility check
Run top-to-bottom before model loading. This notebook performs diagnostics only and writes explicit compatibility reports to `artifacts/`.

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

ARTIFACT_DIR = Path(os.environ.get('AASP_ARTIFACT_DIR', 'artifacts'))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MOUNT_DRIVE = os.environ.get('AASP_MOUNT_DRIVE', '0') == '1'
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted.')
    except ImportError:
        print('AASP_MOUNT_DRIVE=1, but this runtime is not Google Colab; continuing without Drive.')
else:
    print('Drive mount disabled. Set AASP_MOUNT_DRIVE=1 to enable it in Colab.')

In [ ]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def command_output(command):
    try:
        return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT, timeout=10).strip()
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        return f'unavailable: {exc}'

memory = {}
try:
    import psutil
    memory['ram_total_bytes'] = psutil.virtual_memory().total
except ImportError:
    page_size = os.sysconf('SC_PAGE_SIZE') if hasattr(os, 'sysconf') else 0
    page_count = os.sysconf('SC_PHYS_PAGES') if hasattr(os, 'sysconf') else 0
    memory['ram_total_bytes'] = page_size * page_count

disk = shutil.disk_usage(Path.cwd())
gpu_records = []
torch_info = {'installed': False, 'cuda_available': False}
try:
    import torch
    torch_info = {
        'installed': True,
        'version': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'cuda_version': torch.version.cuda,
    }
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            free_bytes, total_bytes = torch.cuda.mem_get_info(index)
            gpu_records.append({
                'index': index,
                'name': props.name,
                'total_vram_bytes': total_bytes,
                'free_vram_bytes': free_bytes,
                'compute_capability': f'{props.major}.{props.minor}',
            })
except ImportError:
    pass

packages = {name: package_version(name) for name in [
    'torch', 'transformers', 'accelerate', 'peft', 'trl', 'bitsandbytes'
]}
environment_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'cuda': torch_info,
    'gpus': gpu_records,
    'gpu_count': len(gpu_records),
    'memory': memory,
    'disk': {'total_bytes': disk.total, 'free_bytes': disk.free},
    'google_drive_mounted': Path('/content/drive/MyDrive').exists(),
    'packages': packages,
    'nvidia_smi': command_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader']),
}
print(json.dumps(environment_report, indent=2))

In [ ]:
MODEL_ID = 'Qwen/Qwen3.8-27B'
PARAMETERS = 27_000_000_000
GIB = 1024 ** 3
available_vram = sum(gpu['free_vram_bytes'] for gpu in gpu_records)

profiles = {
    'BF16': {'weight_bytes': PARAMETERS * 2, 'runtime_factor': 1.20, 'kv_cache_bytes': 4 * GIB},
    'FP16': {'weight_bytes': PARAMETERS * 2, 'runtime_factor': 1.20, 'kv_cache_bytes': 4 * GIB},
    'INT8': {'weight_bytes': PARAMETERS, 'runtime_factor': 1.25, 'kv_cache_bytes': 4 * GIB},
    'INT4': {'weight_bytes': PARAMETERS * 0.5, 'runtime_factor': 1.35, 'kv_cache_bytes': 4 * GIB},
}
compatibility = {}
for name, profile in profiles.items():
    estimated = int(profile['weight_bytes'] * profile['runtime_factor'] + profile['kv_cache_bytes'])
    if available_vram >= estimated:
        status = 'LIKELY_VIABLE'
    elif name == 'INT4' and available_vram >= 14 * GIB:
        status = 'POSSIBLE_LIMITED_INFERENCE'
    else:
        status = 'NOT_VIABLE'
    compatibility[name] = {
        'status': status,
        'estimated_required_vram_bytes': estimated,
        'estimated_required_vram_gib': round(estimated / GIB, 2),
        'assumptions': 'Weights plus format overhead and a 4 GiB KV-cache/runtime allowance; actual use depends on implementation and context length.',
    }

recommendation = (
    'No CUDA GPU detected. Use a rented 24/40/80 GiB GPU or Colab runtime; do not attempt model loading here.'
    if not gpu_records else
    'Use only a profile marked LIKELY_VIABLE. POSSIBLE_LIMITED_INFERENCE requires reduced context and measured loading.'
)
compatibility_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'requested_model': MODEL_ID,
    'parameter_estimate': PARAMETERS,
    'available_vram_bytes': available_vram,
    'available_vram_gib': round(available_vram / GIB, 2),
    'profiles': compatibility,
    'recommendation': recommendation,
    'model_load_attempted': False,
}
(ARTIFACT_DIR / 'environment_report.json').write_text(json.dumps(environment_report, indent=2), encoding='utf-8')
(ARTIFACT_DIR / 'model_compatibility_report.json').write_text(json.dumps(compatibility_report, indent=2), encoding='utf-8')
for name, result in compatibility.items():
    print(f'{name}: {result["status"]} (estimated {result["estimated_required_vram_gib"]} GiB)')
print(recommendation)
for path in [ARTIFACT_DIR / 'environment_report.json', ARTIFACT_DIR / 'model_compatibility_report.json']:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f'Artifact validation failed: {path}')
    print(f'Validated artifact: {path.resolve()}')